# Práctica 4: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

### Ejercicio 1

La biblioteca `Unified Planning` permite leer un dominio de planificación automática a partir de un fichero PDDL y crear posteriormente instancias de ese dominio mediante la interfaz proporcionada por la biblioteca.

En este ejercicio se pretende seguir esa metodología para crear instancias del mundo de los bloques que contengan los bloques $B_{0}$ a $B_{N - 1}$, apilados inicialmente en ese orden, y en la que el objetivo sea que estén apilados al contrario, con el bloque $B_{N - 1}$ sobre la mesa, el bloque $B_{N - 2}$ sobre el $B_{N - 1}$, el $B_{N - 3}$ sobre el $B_{N - 2}$, etc.

Se pide realizar lo siguiente:

1. Leer el dominio del mundo de los bloques a partir del fichero `dominio_mundo_bloques.pddl`.

In [1]:
from unified_planning.io import PDDLReader

In [2]:
lector_PDDL = PDDLReader()
dominio_mundo_bloques = lector_PDDL.parse_problem("dominio_mundo_bloques.pddl", None)

In [3]:
print(dominio_mundo_bloques)

problem name = dominio_mundo_bloques

types = [object]

fluents = [
  bool sobre_la_mesa[b=object]
  bool sobre[b1=object, b2=object]
  bool agarrado[b=object]
  bool brazo_libre
  bool despejado[b=object]
]

actions = [
  action agarrar(object b) {
    preconditions = [
      (sobre_la_mesa(b) and despejado(b) and brazo_libre)
    ]
    effects = [
      sobre_la_mesa(b) := false
      despejado(b) := false
      brazo_libre := false
      agarrado(b) := true
    ]
  }
  action bajar(object b) {
    preconditions = [
      agarrado(b)
    ]
    effects = [
      agarrado(b) := false
      sobre_la_mesa(b) := true
      despejado(b) := true
      brazo_libre := true
    ]
  }
  action desapilar(object b1, object b2) {
    preconditions = [
      (sobre(b1, b2) and despejado(b1) and brazo_libre)
    ]
    effects = [
      sobre(b1, b2) := false
      despejado(b1) := false
      brazo_libre := false
      agarrado(b1) := true
      despejado(b2) := true
    ]
  }
  action apilar(object

2. Completar la definición de la función `crea_instancia_mundo_bloques`, sustituyendo los `...` por código adecuado para que, dado el número `N` de bloques, la función proporcione el problema del mundo de los bloques descrito anteriormente.

In [4]:
from unified_planning.shortcuts import *

In [63]:
def crea_instancia_mundo_bloques(N):
    instancia = dominio_mundo_bloques.clone()  # Trabajamos con una copia del dominio
    # Se añaden los objetos de la instancia
    tipo_objeto = instancia.user_type('object')

    bloques = [Object(f'B{i}', tipo_objeto) for i in range(N)]
    
    # limpiar objetos previos
    instancia._objects = []
    # Añadir los nuevos
    instancia.add_objects(bloques)
    
    # Se establece el estado inicial de la instancia
    sobre_la_mesa = dominio_mundo_bloques.fluent('sobre_la_mesa')
    sobre = dominio_mundo_bloques.fluent('sobre')
    agarrado = dominio_mundo_bloques.fluent('agarrado')
    brazo_libre = dominio_mundo_bloques.fluent('brazo_libre')
    despejado = dominio_mundo_bloques.fluent('despejado')
    #...

    # Estado Inicial
    # 1. B0 sobre la mesa
    instancia.set_initial_value(sobre_la_mesa(bloques[0]), True)
    # 2. Bloques apilados: B0, B1, B2, B3
    for i in range(N-1):
        instancia.set_initial_value(sobre(bloques[i+1],bloques[i]), True)
    # 3. Último bloque no tiene otro apilado
    instancia.set_initial_value(despejado(bloques[N-1]), True)
    # 4. Brazo libre
    instancia.set_initial_value(brazo_libre,True)
    
    # Se establece el objetivo de la instancia
    # 1. Bloques apilados en orden inverso (Bn-1,....B0)
    for i in range(N-1):
        instancia.add_goal(sobre(bloques[i],bloques[i+1]))
    # 2. Bloque Bn-1 sobre la mesa
    instancia.set_initial_value(sobre_la_mesa(bloques[N-1]), True) 
        
    return instancia


crea_instancia_mundo_bloques(5)

problem name = dominio_mundo_bloques

types = [object]

fluents = [
  bool sobre_la_mesa[b=object]
  bool sobre[b1=object, b2=object]
  bool agarrado[b=object]
  bool brazo_libre
  bool despejado[b=object]
]

actions = [
  action agarrar(object b) {
    preconditions = [
      (sobre_la_mesa(b) and despejado(b) and brazo_libre)
    ]
    effects = [
      sobre_la_mesa(b) := false
      despejado(b) := false
      brazo_libre := false
      agarrado(b) := true
    ]
  }
  action bajar(object b) {
    preconditions = [
      agarrado(b)
    ]
    effects = [
      agarrado(b) := false
      sobre_la_mesa(b) := true
      despejado(b) := true
      brazo_libre := true
    ]
  }
  action desapilar(object b1, object b2) {
    preconditions = [
      (sobre(b1, b2) and despejado(b1) and brazo_libre)
    ]
    effects = [
      sobre(b1, b2) := false
      despejado(b1) := false
      brazo_libre := false
      agarrado(b1) := true
      despejado(b2) := true
    ]
  }
  action apilar(object

3. Usar el planificador `Fast Downward` para tratar de resolver la instancia con el mayor número posible de bloques.

In [65]:
problem = crea_instancia_mundo_bloques(15)
planificador = OneshotPlanner(name='fast-downward')
planificador.supports(problem.kind)

resultado = planificador.solve(problem)

print(resultado)


  *** Credits ***
  * In operation mode `OneshotPlanner` at line 2 of `/tmp/ipykernel_248924/3382489591.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    desapilar(B14, B13)
    bajar(B14)
    desapilar(B13, B12)
    apilar(B13, B14)
    desapilar(B12, B11)
    apilar(B12, B13)
    desapilar(B11, B10)
    apilar(B11, B12)
    desapilar(B10, B9)
    apilar(B10, B11)
    desapilar(B9, B8)
    apilar(B9, B10)
    desapilar(B8, B7)
    apilar(B8, B9)
    desapilar(B7, B6)
    apilar(B7, B8)
    desapilar(B6, B5)
    apilar(B6, B7)
    desapilar(B5, B4)
    apilar(B5, B6)
    desapilar(B4, B3)
    apilar(B4, B5)
    desapilar(B3, B2)
    apilar(B3, B4)
    desapilar(B2, B1)
    apilar(B2, B3)
   

### Ejercicio 2

En el marco de la _Conferencia Internacional sobre Planificación Automática y Planificación Temporal_ ([International Conference on Automated Planning and
Scheduling, ICAPS](http://www.icaps-conference.org/)) se celebra, con periodicidad aproximadamente trienal, la _Competición Internacional de Planificación_ (https://www.icaps-conference.org/competitions/).

Esta competición tiene diferentes objetivos: realizar una comparación empírica del estado del arte de los sistemas de planificación; destacar desafíos para la comunidad de Planificación Automática; proponer nuevas direcciones para la investigación y nuevos vínculos con otros campos de la Inteligencia Artificial; y proporcionar nuevos conjuntos de datos que puedan ser utilizados por la comunidad científica como puntos de referencia.

Uno de los dominios utilizados en la competición del año 2002 combinaba el mundo de los bloques con la distribución logística de cajas. En este dominio hay una serie de camiones (que asumimos con capacidad infinita) que transportan cajas entre distintos lugares (que asumimos que están todos conectados entre sí); en esos lugares hay unos palés, sobre los que las cajas se colocan apiladas; los apilamientos se realizan con [polipastos](https://es.wikipedia.org/wiki/Polipasto) (hay al menos uno en cada lugar).

En este ejercicio se pide completar la especificación del dominio que se proporciona a continuación, sustituyendo en las acciones los `...` por hechos adecuados, y tratar de resolver la mayor cantidad posible de las instancias de problemas proporcionadas en la carpeta Depot.

In [66]:
from unified_planning.shortcuts import *

In [67]:
dominio_depot = Problem('Depot')

In [68]:
# Jerarquía de tipos de objetos

Place = UserType('Place')  # Lugar
Locatable = UserType('Locatable')  # Ubicable
Depot = UserType('Depot', Place)  # Almacén (Tipo de lugar)
Distributor = UserType('Distributor', Place)  # Distribuidor (Tipo de lugar)
Truck = UserType('Truck', Locatable)  # Camión (Tipo de ubicable)
Hoist = UserType('Hoist', Locatable)  # Polipasto (Tipo de ubicable)
Surface = UserType('Surface', Locatable)  # Superficie (Tipo de ubicable)
Pallet = UserType('Pallet', Surface)  # Palé (Tipo de superficie)
Crate = UserType('Crate', Surface)  # Caja (Tipo de superficie)

for tipo_de_objeto in [Place, Locatable, Depot, Distributor, Truck, Hoist, Surface, Pallet, Crate]:
    dominio_depot.user_types.append(tipo_de_objeto)

In [69]:
# Predicados booleanos

# El predicado AT representa que el ubicable x está en el lugar y
at = Fluent('AT', BoolType(), x=Locatable, y=Place)
# El predicado ON representa que la caja x está sobre la superficie y
on = Fluent('ON', BoolType(), x=Crate, y=Surface)
# El predicado IN representa que la caja x está en el camión y
# (Nótese el guión bajo incluido en el nombre de la variable, ya que no se
# puede usar in, al tratarse de un identificador reservado de Python)
in_ = Fluent('IN', BoolType(), x=Crate, y=Truck)
# El predicado LIFTING representa que el polipasto x está levantando la caja y
lifting = Fluent('LIFTING', BoolType(), x=Hoist, y=Crate)
# El predicado AVAILABLE representa que el polipasto x está disponible
available = Fluent('AVAILABLE', BoolType(), x=Hoist)
# El predicado CLEAR representa que la superficie x está despejada
clear = Fluent('CLEAR', BoolType(), x=Surface)

for fluente in [at, on, in_, lifting, available, clear]:
    dominio_depot.add_fluent(fluente, default_initial_value=False)

In [72]:
# Esquemas de acciones

# La acción DRIVE representa que el camión x va del lugar y al lugar z
drive = InstantaneousAction('DRIVE', x=Truck, y=Place, z=Place)
x = drive.x
y = drive.y
z = drive.z
for hecho in [at(x,y)]:
    drive.add_precondition(hecho)
for hecho in [at(x,y)]:
    drive.add_effect(hecho, False)
for hecho in [at(x,z)]:
    drive.add_effect(hecho, True)

# La acción LIFT representa que el polipasto x levanta la caja y que se
# encontraba sobre la superficie z en el lugar p
lift = InstantaneousAction('LIFT', x=Hoist, y=Crate, z=Surface, p=Place)
x = lift.x
y = lift.y
z = lift.z
p = lift.p
for hecho in [available(x), on(y,z), at(z,p)]:
    lift.add_precondition(hecho)
for hecho in [on(y,z), available(x)]:
    lift.add_effect(hecho, False)
for hecho in [lifting(x,y)]:
    lift.add_effect(hecho, True)

# La acción DROP representa que el polipasto x deja la caja y sobre la
# superficie z en el lugar p
drop = InstantaneousAction('DROP', x=Hoist, y=Crate, z=Surface, p=Place)
x = drop.x
y = drop.y
z = drop.z
p = drop.p
for hecho in [lifting(x,y), at(x,p), at(z,p)]:
    drop.add_precondition(hecho)
for hecho in [lifting(x,y)]:
    drop.add_effect(hecho, False)
for hecho in [available(x),on(y,z)]:
    drop.add_effect(hecho, True)

# La acción LOAD representa que el polipasto x carga la caja y en el
# camión z en el lugar p
load = InstantaneousAction('LOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = load.x
y = load.y
z = load.z
p = load.p
for hecho in [lifting(x,y), at(x,p), at(z,p)]:
    load.add_precondition(hecho)
for hecho in [lifting(x,y)]:
    load.add_effect(hecho, False)
for hecho in [available(x), in_(y,z)]:
    load.add_effect(hecho, True)

# La acción UNLOAD representa que el polipasto x descarga la caja y del
# camión z en el lugar p
unload = InstantaneousAction('UNLOAD', x=Hoist, y=Crate, z=Truck, p=Place)
x = unload.x
y = unload.y
z = unload.z
p = unload.p
for hecho in [available(x), in_(y,z), at(z,p), at(x,p)]:
    unload.add_precondition(hecho)
for hecho in [available(x), in_(y,z)]:
    unload.add_effect(hecho, False)
for hecho in [at(y,p)]:
    unload.add_effect(hecho, True)

dominio_depot.add_actions([drive, lift, drop, load, unload])

In [98]:


from unified_planning.io import PDDLWriter

path_dominio_problem = 'Depot/dominio_problem.pddl'
escritor = PDDLWriter(dominio_depot)
escritor.write_domain(path_dominio_problem)
planificador = OneshotPlanner(name='fast-downward', params={'fast_downward_search_config': 'astar(hmax())'})
lector = PDDLReader()

numero_instancias = 22;

instancias = [ f'Depot/pfile{i}' for i in range(1 , numero_instancias + 1) ]
for instancia in instancias:
    print('-' * 100)
    print(f'Instancia => {instancia}')
    print('-' * 100) 

    problema = lector.parse_problem(path_dominio_problem,instancia)
    if (planificador.supports(problem_kind=problema.kind)):
        resultado = planificador.solve(problema, timeout=20)
        print(resultado)
    else:
        print(f'Instancia no compatible con el planificador')

    








  *** Credits ***
  * In operation mode `OneshotPlanner` at line 6 of `/tmp/ipykernel_248924/1239224678.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

----------------------------------------------------------------------------------------------------
Instancia => Depot/pfile1
----------------------------------------------------------------------------------------------------
status: SOLVED_SATISFICING
engine: Fast Downward
plan: SequentialPlan:
    lift(hoist1, crate1, pallet0, depot0)
    lift(hoist2, crate0, pallet1, distributor0)
    drop(hoist2, crate0, pallet2, distributor1)
    drop(hoist1, crate1, pallet1, distributor0)
----------------------------------------------------------------------------------------------------
Instancia => Depot/pfile2
---------

### Ejercicio 3

[Sokoban](https://en.wikipedia.org/wiki/Sokoban) es un videojuego clásico de tipo puzle. En este juego el objetivo es empujar cajas, u otro tipo de objetos, en un almacén hasta llevarlos a las ubicaciones de almacenamiento. El juego se ve desde una perspectiva cenital. Los objetos solo se pueden empujar, nunca tirar de ellos, y solo un objeto se puede empujar a la vez. El desafío principal es planificar movimientos correctamente para evitar causar un punto muerto, una situación en la que un objeto o el jugador queda atrapado permanentemente, haciendo que el rompecabezas sea irresoluble.

En este ejercicio se pide lo siguiente:

1. Construir un dominio de planificación automática para el juego del Sokoban. Ese dominio debe contener los siguientes elementos:
   * Tipos de objetos: `thing`, `location`, `direction`, `player` (subtipo de `thing`), `stone` (subtipo de `thing`).
   * Predicados:
     * `CLEAR`: representa que una determinada localización (`location`) no contiene ninguna cosa (`thing`).
     * `AT`: representa que una cosa (`thing`) está en una determinada localización (`location`).
     * `AT-GOAL`: representa que una piedra (`stone`) está en una localización objetivo.
     * `IS-GOAL`: representa que una determinada localización (`location`) es una localización objetivo.
     * `IS-NONGOAL`: representa que una determinada localización (`location`) no es una localización objetivo.
     * `MOVE-DIR`: representa que se puede pasar de una determinada localización (`location`) a otra localización (`location`) adyacente moviéndose en una cierta dirección (`direction`).
   * Acciones:
     * `MOVE`: representa que el jugador (`player`) se mueve de la localización (`location`) que ocupa a una localización (`location`) libre adyacente en una determinada dirección (`direction`).
     * `PUSH-TO-NONGOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que no es una localización objetivo, en una determinada dirección (`direction`).
     * `PUSH-TO-GOAL`: representa que el jugador (`player`), estando en una determinada localización (`location`), empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente, que es una localización objetivo, en una determinada dirección (`direction`).

In [117]:

# Dominio
# 1. Tipos de objetos

thing = UserType('Thing')
location = UserType('Location')
direction = UserType('Direction')
player = UserType('Player', father=thing)
stone = UserType('Stone', father=thing)


# 2. fluents (predicados)

clear = Fluent('CLEAR', BoolType(), l=location)
at = Fluent('AT', BoolType(), t=thing, l=location)
# AT-GOAL en las instancias Sokoban es unario: (at-goal stone-XX)
at_goal = Fluent('AT-GOAL', BoolType(), s=stone)
is_goal = Fluent('IS-GOAL', BoolType(), l=location)
is_non_goal = Fluent('IS-NONGOAL', BoolType(), l_no=location)
move_dir = Fluent('MOVE-DIR', BoolType(), l1=location, l2=location, d=direction)  # se puede pasar de l1 a l2 en una determinada dirección

# 3. Acciones
# -- acción MOVE --
accion_move = InstantaneousAction('MOVE', p=player, l1=location, l2=location, d=direction)
p = accion_move.p
l1 = accion_move.l1
l2 = accion_move.l2
d = accion_move.d

# - Precondiciones
for hecho in [at(p, l1), move_dir(l1, l2, d)]:
    accion_move.add_precondition(hecho)
# - lista de borrado
for hecho in [at(p, l1)]:
    accion_move.add_effect(hecho, False)

# - lista de adición
for hecho in [at(p, l2)]:
    accion_move.add_effect(hecho, True)


# PUSH-TO-NOGOAL => representa que el jugador (`player`), estando en una determinada localización (`location`),
# empuja una piedra (`stone`) desde una localización (`location`) a otra localización (`location`) libre adyacente,
# que no es una localización objetivo, en una determinada dirección (`direction`).
# --- acción PUSH-TO-NOGOAL ---
accion_push_to_nogoal = InstantaneousAction('PUSH-TO-NOGOAL', p=player, l1=location, s=stone, l2=location, d=direction)
p = accion_push_to_nogoal.p
l1 = accion_push_to_nogoal.l1
s = accion_push_to_nogoal.s
l2 = accion_push_to_nogoal.l2
d = accion_push_to_nogoal.d

# - precondiciones
for hecho in [at(p, l1), at(s, l1), move_dir(l1, l2, d), clear(l2), is_non_goal(l2), is_non_goal(l1)]:
    accion_push_to_nogoal.add_precondition(hecho)
# - lista de borrado
for hecho in [at(p, l1), at(s, l1), clear(l2), at_goal(s)]:
    accion_push_to_nogoal.add_effect(hecho, False)
# - lista de adición
for hecho in [at(p, l2), at(s, l2)]:
    accion_push_to_nogoal.add_effect(hecho, True)

# PUSH-TO-GOAL: empuja una piedra a una localización objetivo
# --- acción PUSH-TO-GOAL ---
accion_push_to_goal = InstantaneousAction('PUSH-TO-GOAL', p=player, l1=location, s=stone, l2=location, d=direction)
p = accion_push_to_goal.p
l1 = accion_push_to_goal.l1
l2 = accion_push_to_goal.l2
s = accion_push_to_goal.s
d = accion_push_to_goal.d

# - precondiciones
for hecho in [at(p, l1), at(s, l1), clear(l2), move_dir(l1, l2, d), is_non_goal(l1), is_goal(l2)]:
    accion_push_to_goal.add_precondition(hecho)

# - lista de borrado
for hecho in [at(p, l1), at(s, l1), clear(l2)]:
    accion_push_to_goal.add_effect(hecho, False)
# - lista de adición
for hecho in [at(p, l2), at(s, l2), at_goal(s)]:
    accion_push_to_goal.add_effect(hecho, True)


# Hasta aquí se define el dominio del problema.
# Lo salvamos a un fichero

# -1 instanciamos el problema
sokoban_problem = Problem('Sokoban problem')

# -2 Añadimos los tipos
for tipo_de_objeto in [thing, location, direction, player, stone]:
    sokoban_problem.user_types.append(tipo_de_objeto)

# -3 Añadimos los predicados
for fluent in [clear, at, at_goal, is_goal, is_non_goal, move_dir]:
    sokoban_problem.add_fluent(fluent, default_initial_value=False)

# -4 Añadimos las acciones
sokoban_problem.add_actions([accion_move, accion_push_to_nogoal, accion_push_to_goal])


2. Usar el algoritmo $\mathrm{A}^{*}$ y la heurística $h^{\mathrm{max}}$ para resolver, con la menor cantidad posible de movimientos de empuje, los puzles que se encuentran en la carpeta Sokoban. Para ello, asignar coste $1$ a las acciones `PUSH-TO-NONGOAL` y `PUSH-TO-GOAL` y coste $0$ al resto de acciones.

In [118]:

import re
from pathlib import Path

# - Añadir función de coste total (limpiar métricas previas para evitar duplicados)
sokoban_problem._metrics = []
sokoban_problem.add_quality_metric(MinimizeActionCosts({
    accion_push_to_nogoal: Int(1),
    accion_push_to_goal: Int(1),
    accion_move: Int(0)
}))

# - Escribimos el dominio a fichero
writer = PDDLWriter(sokoban_problem)
writer.write_domain('Sokoban/sokoban_problem.pddl')

planificador_sokoban = OneshotPlanner(name='fast-downward', params={'fast_downward_search_config': 'astar(hmax())'})
directorio = Path("./Sokoban")
reader = PDDLReader()

# Leemos el dominio generado como string (tiene MinimizeActionCosts)
with open('Sokoban/sokoban_problem.pddl') as f:
    dominio_str = f.read()

instancias = sorted([f.name for f in directorio.iterdir() if f.is_file() and f.name.startswith("p")])
print(instancias)

for instancia in instancias:
    print('-' * 100)
    print(f'Instancia => {instancia}')
    print('-' * 100) 
    path_instancia = f'./Sokoban/{instancia}'
    
    with open(path_instancia) as f:
        instancia_str = f.read()
    
    # Los ficheros de instancia usan sintaxis PDDL 2.1 (total-cost) que UP no puede parsear.
    # Eliminamos esas líneas porque los costes ya están definidos en el dominio Python
    # mediante MinimizeActionCosts → la optimización de costes se mantiene.
    instancia_str = re.sub(r'\(increase\s*\(total-cost\)[^)]*\)', '', instancia_str)
    instancia_str = re.sub(r'\(:metric\s+minimize\s+\(total-cost\)\)', '', instancia_str)
    
    problema = reader.parse_problem_string(dominio_str, instancia_str)
    if planificador_sokoban.supports(problem_kind=problema.kind):
        resultado = planificador_sokoban.solve(problema, timeout=20)
        print(resultado)
    else:
        print(f'Instancia no compatible con el planificador')


  *** Credits ***
  * In operation mode `OneshotPlanner` at line 16 of `/tmp/ipykernel_248924/4107668846.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

['p01.pddl', 'p02.pddl', 'p03.pddl', 'p04.pddl', 'p05.pddl', 'p06.pddl', 'p07.pddl', 'p08.pddl', 'p09.pddl', 'p10.pddl', 'p11.pddl', 'p12.pddl', 'p13.pddl', 'p14.pddl', 'p15.pddl', 'p16.pddl', 'p17.pddl', 'p18.pddl', 'p19.pddl', 'p20.pddl', 'p21.pddl', 'p22.pddl', 'p23.pddl', 'p24.pddl', 'p25.pddl', 'p26.pddl', 'p27.pddl', 'p28.pddl', 'p29.pddl', 'p30.pddl']
----------------------------------------------------------------------------------------------------
Instancia => p01.pddl
----------------------------------------------------------------------------------------------------
Instancia no compatible con el plan